# Data Curation with values

## Setup

In [ ]:
import chromadb
import pandas as pd
import os
# set which GPU to use
#os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [ ]:
target_file = "../data/heal_data/master_sde.jsonl"

target_df = pd.read_json(target_file, lines=True)

In [ ]:
target_df

In [ ]:
def format_name_type_and_desc(row):
    return f"{row['sde_name']} ({row['sde_data_type']}): {row['sde_description']}"

In [ ]:
target_df["name_type_and_desc"] = target_df.apply(format_name_type_and_desc, axis=1)

In [ ]:
target_df["name_type_and_desc"].iloc[0]

In [ ]:
target_df.head(n=3)

***Push to a collection***
- embed name_type_and_desc using bge

In [ ]:
from FlagEmbedding import FlagModel
# base model
#model = FlagModel('BAAI/bge-large-en-v1.5', use_fp16=True)
# custom embedding model
model = FlagModel('uc-ctds/bge-large-en-v1.5-bio-mapping', use_fp16=True)
models = {
    "name_type_and_desc": model
}

In [ ]:
embeddings = {
    "name_type_and_desc": models["name_type_and_desc"].encode(target_df["name_type_and_desc"].tolist())
}

In [ ]:
len(embeddings["name_type_and_desc"])

In [ ]:
client = chromadb.Client()

In [ ]:
try:
    # delete collection if already exists
    client.delete_collection(name='prop_name_desc')
except Exception:
    print('collection does not exist, do nothing')

In [ ]:
collection = client.create_collection("prop_name_desc", metadata={"hnsw:space": "cosine"})

In [ ]:
batch_size = 5000
for i in range(0, target_df.shape[0], batch_size):
    print(f'adding records to collection, from {i} to {i+batch_size}')
    # just supply a list of embeddings and metadata to chroma
    # see https://docs.trychroma.com/docs/collections/add-data
    collection.add(
        embeddings=embeddings["name_type_and_desc"][i:i+batch_size].tolist(),
        ids=[str(id) for id in target_df.index[i:i+batch_size].tolist()]
        #metadatas=target_df.to_dict("records")[i:i+batch_size]
    )

Eval

In [ ]:
def get_top_k(query_combined_emb, k):
    results = collection.query(
        query_embeddings=query_combined_emb,
        n_results=k
    )
    return results

In [ ]:
def return_top_k_results(row, k):
    format_name_type_and_desc = f"{row['field_name']} ({row['field_type']}): {row['field_description']}"
    query_combined_emb = models["name_type_and_desc"].encode(format_name_type_and_desc)
    results = get_top_k(query_combined_emb, k)
    return results

In [ ]:
def index_to_name(index_list):
    name_list = []
    for index in index_list:
        name = target_df.loc[int(index)]["sde_name"]
        name_list.append(name)
    return name_list

In [ ]:
def format_results(results):
    formatted_results_df = pd.DataFrame({
        'ids': results['ids'][0],
        'distances': results['distances'][0]
    })
    formatted_results_df.sort_values(by='distances', ascending=True, inplace=True)
    formatted_results_df['names'] = index_to_name(formatted_results_df['ids'])
    return pd.Series([
        formatted_results_df['ids'].to_list(),
        formatted_results_df['distances'].to_list(),
        formatted_results_df['names'].to_list()
    ])

In [ ]:
def process_benchmark(benchmark_name):
    print(f'processing {benchmark_name}')
    df = pd.read_csv(benchmark_name, sep='\t')
    df = df.dropna(subset=["field_name", "field_description", "element_title", "element_description"])
    # embed query variables -- var name, var desc and return top_k
    # by searching CDE embeddings
    top_k_list = [1, 5, 10]
    for k in top_k_list:
        col_name = f'top_{k}_results'
        df[col_name] = df.apply(lambda x: return_top_k_results(x, k=k), axis=1)
        # format results
        output_cols = f'top_{k}_ids,top_{k}_distances,top_{k}_names'.split(',')
        df[output_cols] = df[col_name].apply(
            lambda x: format_results(x)
        )
    print('returning top k results')
    return df


In [ ]:
def calculate_acc(truth, pred):
  correct = 0
  for t, p_list_of_names in zip(truth, pred):
      if t in p_list_of_names:
          correct += 1
  return correct / len(truth)

In [ ]:
def run_evals(df, benchmark_name, embedding_model):
    # print('calculating metrics')
    evals = {}
    row_index = []
    top_k_list = [1, 5, 10]
    for k in top_k_list:
        evals[f'accuracy_{k}'] = []

    row_index.append(f'{benchmark_name}_{df.shape[0]}_{embedding_model}')
    for k in top_k_list:
        col_name = f'top_{k}_names'
        top_k_names_list = df[col_name].to_list()
        truth_list = df['element_title'].to_list()
        accuracy = calculate_acc(truth_list, top_k_names_list)
        evals[f'accuracy_{k}'].append(accuracy)

    # print('returning metrics')
    return pd.DataFrame(evals, index=row_index)

In [ ]:
def run_evals_per_row(row, k):
    col_name = f'top_{k}_names'
    top_k_names_list = row[col_name]
    truth = row['element_title']
    is_match = truth in top_k_names_list
    return is_match

In [ ]:
def get_metrics_per_row(df):
    top_k_list = [5]
    for k in top_k_list:
        output_cols = f'is_match_in_top_{k}_name_desc'
        df[output_cols] = df.apply(lambda x: run_evals_per_row(x, k), axis=1)
    return df

In [ ]:
benchmark_name = '../data/heal_data/HEAL_CDE_Mappings-HDP00895-FILTERED.tsv'
embedding_model = 'bge'
df = process_benchmark(benchmark_name=benchmark_name)
metrics_per_row = get_metrics_per_row(df)

In [ ]:
df.columns

In [ ]:
metrics_per_row['is_match_in_top_5_name_desc'].value_counts()

In [ ]:
metrics_per_row.columns

In [ ]:
metrics_per_row.shape

In [ ]:
cols_to_keep = [
    'field_name', 'field_description', 'field_type',
    'element_title', 'element_description', 'top_5_results', 'top_5_ids', 'top_5_distances',
    'top_5_names',  'is_match_in_top_5_name_desc'
    ]

In [ ]:
metrics_per_row.to_csv('../data/metrics_per_row.csv', sep='\t', columns=cols_to_keep)

In [ ]:
benchmark_names = ['../data/heal_data/HEAL_CDE_Mappings-HDP00895-FILTERED.tsv']
embedding_model_name = 'bge'
results = pd.concat([
    run_evals(df=process_benchmark(eval_data), benchmark_name=eval_data, embedding_model=embedding_model_name)
    for eval_data in benchmark_names
])

In [ ]:
results